# Catalog API playground (control-plane sandbox)

> **This is a control-plane sandbox — not a lesson.** The *lessons* (CAT-1, CAT-2, CAT-3, CAT-5) stick to SQL. This notebook is where the raw HTTP plumbing lives: OAuth token flows, Nessie's REST ref surface, Polaris's management API, common gotchas (401 vs 403 vs 404, realm headers). Read it, poke it, break it — none of the cells depend on each other.

Sections:

1. **Nessie REST** — list refs, create/merge/delete a branch by hand; contrast with the [CAT-3](./nessie/cat3_branching.ipynb) SQL surface.
2. **Polaris admin API** — OAuth2 client-credentials flow, list principals, list catalog-roles, grant a privilege, list namespaces as that principal (the plumbing behind [CAT-2](cat2_polaris_rbac.ipynb)).
3. **Common gotchas** — the four failure shapes you'll actually hit.

**Prereqs:** `make up && make catalogs-up`. Nessie on `:19120`, Polaris on `:8181`.

**Everything is stdlib `urllib`.** No SDK, no boto3, no hidden magic — the shape of every call is verbatim what you'd see in a real HTTP client / cURL / Postman.

In [ ]:
import json
import urllib.error
import urllib.parse
import urllib.request

NESSIE = "http://localhost:19120"
POLARIS = "http://localhost:8181"
REALM = "default"


def http(method, url, headers=None, body=None, form=None):
    """One tiny HTTP helper. Returns (status, body-dict, response-headers-dict).
    Never raises on non-2xx — inspecting error bodies IS the point of this notebook."""
    h = dict(headers or {})
    data = None
    if form is not None:
        h["Content-Type"] = "application/x-www-form-urlencoded"
        data = urllib.parse.urlencode(form).encode()
    elif body is not None:
        h["Content-Type"] = "application/json"
        data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method=method, headers=h)
    try:
        with urllib.request.urlopen(req) as r:
            raw, status, resp_headers = r.read(), r.status, dict(r.headers)
    except urllib.error.HTTPError as e:
        raw, status, resp_headers = e.read(), e.code, dict(e.headers)
    try:
        parsed = json.loads(raw) if raw else {}
    except json.JSONDecodeError:
        parsed = {"_raw": raw.decode(errors="replace")}
    return status, parsed, resp_headers


def snippet(body, n=280):
    s = json.dumps(body, indent=2) if body else "(empty)"
    return s if len(s) <= n else s[:n] + " …[truncated]"

print("helpers ready")

## 1. Nessie REST — refs by hand

Nessie's REST API is versioned under `/api/v2`. The endpoints we exercise here:

| Method | Path | Purpose |
|---|---|---|
| `GET`  | `/api/v2/trees` | List all refs (branches + tags) |
| `GET`  | `/api/v2/trees/{ref}` | Get a specific ref's current hash |
| `POST` | `/api/v2/trees?name=<b>&type=BRANCH&sourceRefName=main` | Create branch |
| `POST` | `/api/v2/trees/{ref}@{hash}/history/merge` | Merge one ref into another |
| `DELETE` | `/api/v2/trees/{ref}@{hash}?type=BRANCH` | Delete a branch |

Notice the `@{hash}` in the write paths: that's Nessie's **optimistic-lock** — the write only succeeds if the ref is still at the hash you supplied. Blind writes are impossible.

The Spark SQL extension you use in [CAT-3](./nessie/cat3_branching.ipynb) ends up calling these same endpoints under the hood.

In [ ]:
# 1a. List all refs.
status, body, _ = http("GET", f"{NESSIE}/api/v2/trees")
print(f"GET /api/v2/trees → {status}")
print("refs:")
for ref in body.get("references", []):
    print(f"   {ref['type']:6}  {ref['name']:20}  hash={ref['hash'][:16]}…")

In [ ]:
# 1b. Create a branch from main. `sourceRefName` is the parent; the body carries
#     the source-ref anti-lock check (Nessie refuses if main has moved since).
status_main, main_body, _ = http("GET", f"{NESSIE}/api/v2/trees/main")
main_hash = main_body["reference"]["hash"]
print(f"main is at {main_hash[:16]}…")

BRANCH = "playground_probe"
# Idempotent — delete if it's already there from a prior run.
st, existing, _ = http("GET", f"{NESSIE}/api/v2/trees/{urllib.parse.quote(BRANCH)}")
if st == 200:
    old = existing["reference"]["hash"]
    http("DELETE", f"{NESSIE}/api/v2/trees/{urllib.parse.quote(BRANCH)}@{old}?type=BRANCH")
    print(f"cleanup: deleted stale {BRANCH}")

q = urllib.parse.urlencode({"name": BRANCH, "type": "BRANCH", "sourceRefName": "main"})
status, body, _ = http(
    "POST",
    f"{NESSIE}/api/v2/trees?{q}",
    body={"type": "BRANCH", "name": "main", "hash": main_hash},
)
print(f"POST /api/v2/trees?name={BRANCH}&type=BRANCH&sourceRefName=main → {status}")
print(f"new ref: {body['reference']['name']} @ {body['reference']['hash'][:16]}…")
assert body["reference"]["hash"] == main_hash, "a new branch should share main's hash at creation"

In [ ]:
# 1c. Delete the branch — with a hash check. Try WITHOUT one first to see the anti-lock
#     behaviour, then include it.
st_no_hash, body_no_hash, _ = http("DELETE", f"{NESSIE}/api/v2/trees/{BRANCH}?type=BRANCH")
print(f"DELETE (no hash)  → {st_no_hash}")
print(f"body: {snippet(body_no_hash, 220)}")

# Now with the current hash — should succeed.
st, _, _ = http("GET", f"{NESSIE}/api/v2/trees/{BRANCH}")
if st == 200:
    _, cur, _ = http("GET", f"{NESSIE}/api/v2/trees/{BRANCH}")
    cur_hash = cur["reference"]["hash"]
    status_del, body_del, _ = http("DELETE", f"{NESSIE}/api/v2/trees/{BRANCH}@{cur_hash}?type=BRANCH")
    print(f"\nDELETE @{cur_hash[:16]}… → {status_del}")
else:
    print("branch already gone (the no-hash DELETE removed it — behaviour depends on Nessie version).")

### Nessie REST vs Nessie SQL extension

| Concern | Raw REST | Spark SQL extension |
|---|---|---|
| `CREATE BRANCH dev` | `POST /api/v2/trees?name=dev&sourceRefName=main` + body | `CREATE BRANCH dev IN nessie_catalog FROM main` |
| Switch active ref | Register a shadow catalog with a different `prefix` | `USE REFERENCE dev IN nessie_catalog` |
| Merge | `POST /api/v2/trees/main@<hash>/history/merge` | `MERGE BRANCH dev INTO main IN nessie_catalog` |
| Idempotent optimistic lock | Hash on write paths (`@<hash>`) | Handled inside the extension |
| Bulk / cross-table branch mgmt | Yes (this is the CI/CD surface) | Only what SQL exposes |

The REST surface is what a **CI/CD pipeline** would call to spin CI branches, delete PR branches on merge, etc. The SQL extension is what a **data engineer** uses inside a Spark job.

## 2. Polaris admin API — OAuth2 + RBAC plumbing

Polaris speaks two APIs:

1. **`/api/catalog/v1/…`** — the standard Iceberg REST catalog (namespaces, tables, config). Same shape any Iceberg-REST-compatible engine speaks.
2. **`/api/management/v1/…`** — Polaris's own admin API (principals, principal-roles, catalog-roles, grants). Not part of the Iceberg REST spec.

Every call needs (a) the `Polaris-Realm` header — realms are the top-level tenant boundary — and (b) an OAuth2 Bearer token. This section fetches a token via the client-credentials grant, then walks the admin surface. This is the plumbing behind [CAT-2](cat2_polaris_rbac.ipynb).

In [ ]:
# 2a. OAuth2 client-credentials grant. This is the OAuth2 spec's grant for
#     service-to-service auth — the caller IS the resource owner and
#     authenticates with its own client_id + client_secret.
status, body, _ = http(
    "POST",
    f"{POLARIS}/api/catalog/v1/oauth/tokens",
    headers={"Polaris-Realm": REALM},
    form={
        "grant_type": "client_credentials",
        "client_id": "root",
        "client_secret": "secret",
        "scope": "PRINCIPAL_ROLE:ALL",
    },
)
print(f"POST /api/catalog/v1/oauth/tokens → {status}")
assert status == 200 and "access_token" in body
token = body["access_token"]
print(f"got a Bearer token ({len(token)} chars, {body.get('token_type')}, expires_in={body.get('expires_in')}s)")
print(f"scope granted: {body.get('scope')}")
auth = {"Authorization": f"Bearer {token}", "Polaris-Realm": REALM}

In [ ]:
# 2b. Walk the admin surface: principals, principal-roles, catalog-roles.
for endpoint in [
    "/api/management/v1/principals",
    "/api/management/v1/principal-roles",
    "/api/management/v1/catalogs",
]:
    status, body, _ = http("GET", f"{POLARIS}{endpoint}", headers=auth)
    label = endpoint.rsplit("/", 1)[-1]
    print(f"GET {endpoint} → {status}")
    print(f"   {label}: {snippet(body, 200)}")
    print()

In [ ]:
# 2c. Walk the grants on the `lab` catalog's `catalog_admin` role — what does
#     "admin on this catalog" actually mean, enum by enum?
status, body, _ = http(
    "GET",
    f"{POLARIS}/api/management/v1/catalogs/lab/catalog-roles/catalog_admin/grants",
    headers=auth,
)
print(f"GET /api/management/v1/catalogs/lab/catalog-roles/catalog_admin/grants → {status}")
print("catalog_admin privileges:")
for g in body.get("grants", []):
    print(f"   - {g['privilege']}")

# ↑ This is the taxonomy CAT-2's 403 body names verbatim. `NAMESPACE_DROP`,
#   `TABLE_DROP`, `TABLE_WRITE_DATA` — all separate privileges, all grepable in
#   the Polaris source when you're figuring out which enum unblocks an op.

In [ ]:
# 2d. Data-plane call: list namespaces via the catalog API (not the mgmt API).
#     This is what any Iceberg-REST client (Spark, Trino, dbt-spark) speaks.
status, body, _ = http(
    "GET", f"{POLARIS}/api/catalog/v1/lab/namespaces", headers=auth,
)
print(f"GET /api/catalog/v1/lab/namespaces → {status}")
print(f"namespaces: {body.get('namespaces', [])}")

# This is the shape a real query engine executes — CAT-2's alice hits this exact
# endpoint (with her own token) to demonstrate that her NAMESPACE_LIST grant is
# preserved after we add NAMESPACE_DROP.

## 3. Common gotchas — the four failure shapes you'll actually hit

Each cell here **intentionally errors** so you learn to recognise the shapes. None of them affect state — they're pure reads with bad inputs.

In [ ]:
# Gotcha 1 — 401 UNAUTHORIZED: missing or invalid Bearer token.
status, body, _ = http("GET", f"{POLARIS}/api/management/v1/principals",
                       headers={"Polaris-Realm": REALM})   # no Authorization
print(f"no token         → HTTP {status}   {snippet(body, 160)}")

status, body, _ = http("GET", f"{POLARIS}/api/management/v1/principals",
                       headers={"Polaris-Realm": REALM, "Authorization": "Bearer garbage"})
print(f"garbage token    → HTTP {status}   {snippet(body, 160)}")
print()
print("    401 = 'I don't know who you are'. Fix: refetch a token via client_credentials.")

In [ ]:
# Gotcha 2 — 403 FORBIDDEN: you're authenticated, but not authorized.
# We fake this by asking Polaris to LIST a resource root doesn't manage.
# (For a full alice-403 demo, see CAT-2.)
#
# The characteristic shape: response body is JSON with error.type = ForbiddenException
# and a human-readable message naming the denied enum. That message is the design.
print("For the canonical 403 example see CAT-2 §Break/Detect — alice's")
print("DELETE against /namespaces/cat2_demo without NAMESPACE_DROP.")
print("Body shape:")
print('  {"error": {"type": "ForbiddenException", "code": 403,')
print('             "message": "Principal \'alice\' with activated PrincipalRoles ...')
print('                         is not authorized for op DROP_NAMESPACE"}}')
print()
print("    403 = 'I know you, but you can't do that'. Fix: grant the exact enum in the message.")

In [ ]:
# Gotcha 3 — 404 NOT FOUND: right URL shape, wrong name.
status, body, _ = http("GET", f"{POLARIS}/api/catalog/v1/lab/namespaces/does_not_exist",
                       headers=auth)
print(f"non-existent namespace  → HTTP {status}   {snippet(body, 200)}")

status, body, _ = http("GET", f"{NESSIE}/api/v2/trees/does_not_exist")
print(f"non-existent Nessie ref → HTTP {status}   {snippet(body, 200)}")
print()
print("    404 = 'that thing doesn't exist'. Fix: check the name — case-sensitive on both APIs.")

In [ ]:
# Gotcha 4 — REALM MISCONFIGURATION. Polaris multiplexes on the `Polaris-Realm`
# header; realms are the top-level tenant boundary. Every principal, role, and
# catalog is scoped to one realm. Two behaviours to know:
#
#   (a) OMITTING the header — this dev bootstrap accepts it and falls back to
#       the default realm. Real production deployments turn on strict-mode and
#       will reject the request. Depending on it in code is a footgun.
#   (b) USING THE WRONG REALM — the token you fetched under realm A will not
#       resolve principals/roles/catalogs in realm B. You'll see a 401 or 404.
#
# We can safely test (a) here:
status, body, _ = http("POST",
                       f"{POLARIS}/api/catalog/v1/oauth/tokens",
                       form={"grant_type": "client_credentials",
                             "client_id": "root", "client_secret": "secret",
                             "scope": "PRINCIPAL_ROLE:ALL"})
print(f"OAuth WITHOUT Polaris-Realm header → HTTP {status}")
print("   (accepted in this dev bootstrap — do NOT rely on this in prod)")

# And (b): fabricate a wrong realm on a mgmt call.
status, body, _ = http("GET",
                       f"{POLARIS}/api/management/v1/principals",
                       headers={"Authorization": f"Bearer {token}",
                                "Polaris-Realm": "nonexistent_realm"})
print(f"\nGET /principals with WRONG realm → HTTP {status}   {snippet(body, 200)}")

print("\n    Always pass `Polaris-Realm: <your realm>`. Correct realm → correct scope.")

## Takeaways

- **Catalog admin work is HTTP work.** There is no `GRANT` SQL in the Iceberg REST spec — RBAC is a control-plane concern, so it's a REST surface, not a query-engine surface.
- **Nessie speaks two protocols** to the same commit graph: `/api/v2/…` (native, used by `NessieCatalog`) and `/iceberg/…` (Iceberg REST adapter, used by Nimtable). Writes and reads through either land in the same ref store.
- **Polaris = Iceberg REST + a management API.** The catalog API is standard; the management API is Polaris's addition and is where every grant / role / principal call lives.
- **401 / 403 / 404 have distinct meanings — don't conflate them.** 401 = *who are you*, 403 = *you can't do that*, 404 = *no such thing*.

**Where to go from here:** the *lessons* — [CAT-1](./nessie/cat1_nessie_intro.ipynb), [CAT-2](cat2_polaris_rbac.ipynb), [CAT-3](./nessie/cat3_branching.ipynb), [CAT-5](cat5_federation.ipynb) — use SQL. This playground exists so those lessons don't have to.